In [ ]:
# =========================================================
# 0. Imports
# =========================================================
import polars as pl
import duckdb
from pathlib import Path

# =========================================================
# 1. Rutas
# =========================================================
EXCEL_PATH = Path(
    r"C:\Users\andres.hernandez\OneDrive - IMECOL S.A.S\Documentos\PRUEBA PI PRECIOS\Lista de precios CNH Consolidadas Vigente.xlsx"
)

EXCEL_DISP_PATH = Path(
    r"C:\Users\andres.hernandez\OneDrive - IMECOL S.A.S\Documentos\PRUEBA PI PRECIOS\Lista Disponibilidad Origen.xlsx"
)

DUCKDB_PATH = Path("precios.duckdb")
CSV_OUT = Path("precios_consolidados.csv")

print("Ruta Excel precios:", EXCEL_PATH.resolve())
print("Existe Excel precios:", EXCEL_PATH.exists())
print("Ruta Excel disponibilidad:", EXCEL_DISP_PATH.resolve())
print("Existe Excel disponibilidad:", EXCEL_DISP_PATH.exists())

if not EXCEL_PATH.exists():
    raise FileNotFoundError(f"No se encontró el archivo de precios: {EXCEL_PATH.resolve()}")

if not EXCEL_DISP_PATH.exists():
    raise FileNotFoundError(f"No se encontró el archivo de disponibilidad: {EXCEL_DISP_PATH.resolve()}")

# =========================================================
# 2. Función para leer precios (A = ref, I = precio)
# =========================================================
def leer_hoja_precio(sheet_name: str, col_precio_out: str) -> pl.DataFrame:
    df = pl.read_excel(
        EXCEL_PATH,
        sheet_name=sheet_name,
        has_header=True
    )

    cols = df.columns

    return (
        df.select([
            pl.col(cols[0])
              .cast(pl.Utf8)
              .str.strip_chars()
              .alias("referencia"),

            pl.col(cols[8])
              .cast(pl.Float64)
              .alias(col_precio_out)
        ])
        .filter(
            pl.col("referencia").is_not_null() &
            (pl.col("referencia") != "")
        )
        .unique(subset=["referencia"], keep="first")
    )

# =========================================================
# 3. Leer precios por origen
# =========================================================
df_usa = leer_hoja_precio("USA", "precio_usa")
df_br  = leer_hoja_precio("BRASIL", "precio_br")
df_eur = leer_hoja_precio("EUR", "precio_eur")

print("Filas USA :", df_usa.height)
print("Filas BR  :", df_br.height)
print("Filas EUR :", df_eur.height)

# =========================================================
# 4. Referencia única global (precios)
# =========================================================
df_refs = (
    pl.concat([
        df_usa.select("referencia"),
        df_br.select("referencia"),
        df_eur.select("referencia")
    ])
    .unique()
)

# =========================================================
# 5. BUSCARV precios (LEFT JOIN)
# =========================================================
df_precios = (
    df_refs
    .join(df_usa, on="referencia", how="left")
    .join(df_br,  on="referencia", how="left")
    .join(df_eur, on="referencia", how="left")
)

# =========================================================
# 6. Leer disponibilidades (A, I, L, M)
# =========================================================
df_disp_raw = pl.read_excel(
    EXCEL_DISP_PATH,
    sheet_name="LISTA AGCS",
    has_header=True
)

cols_disp = df_disp_raw.columns

df_disp = (
    df_disp_raw.select([
        pl.col(cols_disp[0])
          .cast(pl.Utf8)
          .str.strip_chars()
          .alias("referencia"),

        pl.col(cols_disp[8])
          .cast(pl.Int64)
          .alias("disp_br"),

        pl.col(cols_disp[11])
          .cast(pl.Int64)
          .alias("disp_eur"),

        pl.col(cols_disp[12])
          .cast(pl.Int64)
          .alias("disp_usa"),
    ])
    .filter(
        pl.col("referencia").is_not_null() &
        (pl.col("referencia") != "")
    )
    .unique(subset=["referencia"], keep="first")
)

print("Filas disponibilidad:", df_disp.height)

# =========================================================
# 7. COMBINAR precios + disponibilidades
# =========================================================
df_final = (
    df_precios
    .join(df_disp, on="referencia", how="left")
    .sort("referencia")
)

print("Total referencias únicas:", df_final.height)
print(df_final.head(5))

# =========================================================
# 8. Guardar en DuckDB + Exportar CSV (CIERRE AUTOMÁTICO)
# =========================================================
with duckdb.connect(DUCKDB_PATH) as con:
    con.register("tmp_precios", df_final.to_pandas())

    con.execute("""
    CREATE OR REPLACE TABLE precios_consolidados AS
    SELECT * FROM tmp_precios
    """)

    con.execute(f"""
    COPY precios_consolidados
    TO '{CSV_OUT}'
    WITH (HEADER, DELIMITER ',');
    """)

    print("\nMuestra de datos:")
    print(con.execute("SELECT * FROM precios_consolidados LIMIT 10").df())

print("✅ Proceso terminado correctamente")
print("Archivos generados:")
print("-", DUCKDB_PATH)
print("-", CSV_OUT)
print("✅ DuckDB cerrado correctamente")

In [1]:
# =========================================================
# 0. Imports
# =========================================================
import polars as pl
import duckdb
from pathlib import Path

# =========================================================
# 1. Rutas
# =========================================================
EXCEL_PATH = Path(
    r"C:\Users\andres.hernandez\OneDrive - IMECOL S.A.S\Documentos\PRUEBA PI PRECIOS\Lista de precios CNH Consolidadas Vigente.xlsx"
)

EXCEL_DISP_PATH = Path(
    r"C:\Users\andres.hernandez\OneDrive - IMECOL S.A.S\Documentos\PRUEBA PI PRECIOS\Lista Disponibilidad Origen.xlsx"
)

DUCKDB_PATH = Path("precios.duckdb")
CSV_OUT = Path("precios_consolidados.csv")

print("Ruta Excel precios:", EXCEL_PATH.resolve())
print("Existe Excel precios:", EXCEL_PATH.exists())
print("Ruta Excel disponibilidad:", EXCEL_DISP_PATH.resolve())
print("Existe Excel disponibilidad:", EXCEL_DISP_PATH.exists())

if not EXCEL_PATH.exists():
    raise FileNotFoundError(f"No se encontró el archivo de precios: {EXCEL_PATH.resolve()}")

if not EXCEL_DISP_PATH.exists():
    raise FileNotFoundError(f"No se encontró el archivo de disponibilidad: {EXCEL_DISP_PATH.resolve()}")

# =========================================================
# 2. Función para leer precios (A = ref, I = precio)
# =========================================================
def leer_hoja_precio(sheet_name: str, col_precio_out: str) -> pl.DataFrame:
    df = pl.read_excel(
        EXCEL_PATH,
        sheet_name=sheet_name,
        has_header=True
    )

    cols = df.columns

    return (
        df.select([
            pl.col(cols[0])
              .cast(pl.Utf8)
              .str.strip_chars()
              .alias("referencia"),

            pl.col(cols[8])
              .cast(pl.Float64)
              .alias(col_precio_out)
        ])
        .filter(
            pl.col("referencia").is_not_null() &
            (pl.col("referencia") != "")
        )
        .unique(subset=["referencia"], keep="first")
    )

# =========================================================
# 3. Leer precios por origen
# =========================================================
df_usa = leer_hoja_precio("USA", "precio_usa")
df_br  = leer_hoja_precio("BRASIL", "precio_br")
df_eur = leer_hoja_precio("EUR", "precio_eur")

print("Filas USA :", df_usa.height)
print("Filas BR  :", df_br.height)
print("Filas EUR :", df_eur.height)

# =========================================================
# 4. Referencia única global (precios)
# =========================================================
df_refs = (
    pl.concat([
        df_usa.select("referencia"),
        df_br.select("referencia"),
        df_eur.select("referencia")
    ])
    .unique()
)

# =========================================================
# 5. BUSCARV precios (LEFT JOIN)
# =========================================================
df_precios = (
    df_refs
    .join(df_usa, on="referencia", how="left")
    .join(df_br,  on="referencia", how="left")
    .join(df_eur, on="referencia", how="left")
)

# =========================================================
# 6. Leer disponibilidades (A, I, L, M)
# =========================================================
df_disp_raw = pl.read_excel(
    EXCEL_DISP_PATH,
    sheet_name="LISTA AGCS",
    has_header=True
)

cols_disp = df_disp_raw.columns

df_disp = (
    df_disp_raw.select([
        pl.col(cols_disp[0])
          .cast(pl.Utf8)
          .str.strip_chars()
          .alias("referencia"),

        pl.col(cols_disp[8])
          .cast(pl.Int64)
          .alias("disp_br"),

        pl.col(cols_disp[11])
          .cast(pl.Int64)
          .alias("disp_eur"),

        pl.col(cols_disp[12])
          .cast(pl.Int64)
          .alias("disp_usa"),
    ])
    .filter(
        pl.col("referencia").is_not_null() &
        (pl.col("referencia") != "")
    )
    .unique(subset=["referencia"], keep="first")
)

print("Filas disponibilidad:", df_disp.height)

# =========================================================
# 7. COMBINAR precios + disponibilidades
# =========================================================
df_final = (
    df_precios
    .join(df_disp, on="referencia", how="left")
    .sort("referencia")
)

print("Total referencias únicas:", df_final.height)
print(df_final.head(5))

# =========================================================
# 8. Guardar en DuckDB + Exportar CSV
# =========================================================
# La base precios.duckdb contendrá la tabla:
#   precios_consolidados
# con:
#   - referencia
#   - precio_usa, precio_br, precio_eur
#   - disp_usa, disp_br, disp_eur
# Es decir: precios + disponibilidades consolidadas por referencia.

with duckdb.connect(DUCKDB_PATH) as con:
    con.register("tmp_precios", df_final.to_pandas())

    con.execute("""
    CREATE OR REPLACE TABLE precios_consolidados AS
    SELECT * FROM tmp_precios
    """)

    con.execute(f"""
    COPY precios_consolidados
    TO '{CSV_OUT}'
    WITH (HEADER, DELIMITER ',');
    """)

    print("\nMuestra de datos:")
    print(con.execute("SELECT * FROM precios_consolidados LIMIT 10").df())

print("✅ Proceso terminado correctamente")
print("Archivos generados:")
print("-", DUCKDB_PATH, "(contiene precios + disponibilidades consolidadas)")
print("-", CSV_OUT, "(exportación plana de la tabla precios_consolidados)")
print("✅ DuckDB cerrado correctamente")

Ruta Excel precios: C:\Users\andres.hernandez\OneDrive - IMECOL S.A.S\Documentos\PRUEBA PI PRECIOS\Lista de precios CNH Consolidadas Vigente.xlsx
Existe Excel precios: True
Ruta Excel disponibilidad: C:\Users\andres.hernandez\OneDrive - IMECOL S.A.S\Documentos\PRUEBA PI PRECIOS\Lista Disponibilidad Origen.xlsx
Existe Excel disponibilidad: True


Could not determine dtype for column 9, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 15, falling back to string
Could not determine dtype for column 17, falling back to string
Could not determine dtype for column 9, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 15, falling back to string
Could not determine dtype for column 9, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 11, falling back to string
Could not determine dtype for column 12, falling back to string
Could not determine dtype for column 17, falling back to string
Could not determine dtype for column 18, falling back to string
Could not determine dtype for column 19, falling back to string


Filas USA : 348518
Filas BR  : 207028
Filas EUR : 342825
Filas disponibilidad: 45146
Total referencias únicas: 448968
shape: (5, 7)
┌──────────────┬────────────┬───────────┬──────────────┬─────────┬──────────┬──────────┐
│ referencia   ┆ precio_usa ┆ precio_br ┆ precio_eur   ┆ disp_br ┆ disp_eur ┆ disp_usa │
│ ---          ┆ ---        ┆ ---       ┆ ---          ┆ ---     ┆ ---      ┆ ---      │
│ str          ┆ f64        ┆ f64       ┆ f64          ┆ i64     ┆ i64      ┆ i64      │
╞══════════════╪════════════╪═══════════╪══════════════╪═════════╪══════════╪══════════╡
│ 000-200-459  ┆ 636.705828 ┆ null      ┆ null         ┆ null    ┆ null     ┆ null     │
│ 0000100029   ┆ null       ┆ 0.23936   ┆ null         ┆ null    ┆ null     ┆ null     │
│ 0000100806   ┆ null       ┆ 4.9742    ┆ null         ┆ null    ┆ null     ┆ null     │
│ 0000101739   ┆ null       ┆ 1.2342    ┆ 1.4702688    ┆ null    ┆ null     ┆ null     │
│ 000011111350 ┆ null       ┆ null      ┆ 61160.101517 ┆ null    ┆ 